## 셀 1 — 라이브러리 불러오기

In [1]:
# ==================================================
# 1. 라이브러리 불러오기
# ==================================================

import cv2
import torch
import torch.nn as nn

from torchvision import transforms

from PIL import Image

import random
from collections import deque

## 셀 2 — CNN 모델 정의

In [2]:
# ==================================================
# 2. CNN 모델 정의
# ==================================================

class CNN(nn.Module):

    def __init__(self):

        super().__init__()


        self.features = nn.Sequential(

            nn.Conv2d(
                3,
                16,
                3,
                padding=1
            ),

            nn.ReLU(),

            nn.MaxPool2d(2),


            nn.Conv2d(
                16,
                32,
                3,
                padding=1
            ),

            nn.ReLU(),

            nn.MaxPool2d(2),


            nn.Conv2d(
                32,
                64,
                3,
                padding=1
            ),

            nn.ReLU(),

            nn.MaxPool2d(2)
        )


        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                64 * 16 * 16,
                128
            ),

            nn.ReLU(),

            nn.Linear(
                128,
                2
            )
        )


    def forward(self, x):

        x = self.features(x)

        x = self.classifier(x)

        return x

## 셀 3 — 부품 목록 만들기

In [3]:
# ==================================================
# 3. 부품 목록
# ==================================================

objects = [
    "nipper",
    "pen",
    "wire_stripper"
]


print("AI가 알고 있는 부품:")

print(objects)

AI가 알고 있는 부품:
['nipper', 'pen', 'wire_stripper']


## 셀 4 — 부품 모델 불러오는 함수

In [4]:
# ==================================================
# 4. 부품별 모델 불러오기
# ==================================================

def load_object_model(
    target
):

    model_path = (
        f"models/{target}_model.pth"
    )


    checkpoint = torch.load(
        model_path,
        map_location="cpu"
    )


    model = CNN()


    model.load_state_dict(
        checkpoint["model_state"]
    )


    model.eval()


    return model

## 셀 5 — 이미지 전처리

In [5]:
# ==================================================
# 5. 이미지 전처리
# ==================================================

transform = transforms.Compose([

    transforms.Resize(
        (128, 128)
    ),

    transforms.ToTensor()
])

## 셀 6 — 게임 설정

In [6]:
# ==================================================
# 6. 게임 설정
# ==================================================

# 목표 부품이라고 인정할 최소 확률
confidence_threshold = 0.85


# 몇 번 연속 맞아야 정답인지
required_count = 10


# 최근 프레임 확률을 저장
probability_history = deque(
    maxlen=5
)


# 현재 연속 정답 횟수
correct_count = 0


# 이미 찾은 물건
completed_objects = []

## 셀 7 — 첫 번째 목표 선택

In [7]:
# ==================================================
# 7. 첫 번째 목표물 랜덤 선택
# ==================================================

remaining_objects = objects.copy()


target = random.choice(
    remaining_objects
)


model = load_object_model(
    target
)


print()

print("================================")

print("       🔎 물건 찾기 게임")

print("================================")

print()

print("AI가 알고 있는 물건:")

print(objects)

print()

print("🎯 찾아야 하는 물건:")

print(f"👉 {target}")

print()


       🔎 물건 찾기 게임

AI가 알고 있는 물건:
['nipper', 'pen', 'wire_stripper']

🎯 찾아야 하는 물건:
👉 nipper



## 셀 8 — 웹캠 실행

In [8]:
# ==================================================
# 8. 웹캠 실행
# ==================================================

camera = cv2.VideoCapture(0)


if not camera.isOpened():

    print("❌ 웹캠을 열 수 없습니다.")

    raise SystemExit

else:

    print("✅ 웹캠이 시작되었습니다.")

✅ 웹캠이 시작되었습니다.


## 셀 9 — 카메라 인식

In [9]:
# ==================================================
# 9. 게임 진행
# ==================================================

while True:

    ret, frame = camera.read()


    if not ret:

        print(
            "❌ 웹캠 화면을 읽을 수 없습니다."
        )

        break


    # --------------------------------------------------
    # 웹캠 화면 크기
    # --------------------------------------------------

    height, width = frame.shape[:2]


    # --------------------------------------------------
    # 중앙 검사 영역
    # --------------------------------------------------

    box_width = int(
        width * 0.60
    )

    box_height = int(
        height * 0.60
    )


    x1 = int(
        (width - box_width) / 2
    )

    y1 = int(
        (height - box_height) / 2
    )

    x2 = x1 + box_width

    y2 = y1 + box_height


    # --------------------------------------------------
    # AI에는 전체 화면을 사용
    # --------------------------------------------------

    rgb = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB
    )


    image = Image.fromarray(
        rgb
    )


    image = transform(
        image
    )


    image = image.unsqueeze(0)


    # --------------------------------------------------
    # AI 예측
    # --------------------------------------------------

    with torch.no_grad():

        output = model(
            image
        )


        probability = torch.softmax(
            output,
            dim=1
        )


        # 0 = 다른 물건
        # 1 = 목표 물건

        other_confidence = (
            probability[0, 0].item()
        )


        target_confidence = (
            probability[0, 1].item()
        )


    # --------------------------------------------------
    # 최근 확률 저장
    # --------------------------------------------------

    probability_history.append(
        target_confidence
    )


    # --------------------------------------------------
    # 최근 5개 평균
    # --------------------------------------------------

    average_confidence = (

        sum(probability_history)
        / len(probability_history)

    )


    # --------------------------------------------------
    # 목표물인지 확인
    # --------------------------------------------------

    if average_confidence >= confidence_threshold:

        correct_count += 1

    else:

        correct_count = 0


    # --------------------------------------------------
    # 검사 영역 표시
    # --------------------------------------------------

    cv2.rectangle(

        frame,

        (x1, y1),

        (x2, y2),

        (255, 255, 0),

        2
    )


    # --------------------------------------------------
    # Target 표시
    # --------------------------------------------------

    cv2.putText(

        frame,

        f"Target: {target}",

        (30, 40),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.8,

        (0, 255, 255),

        2
    )


    # --------------------------------------------------
    # 목표 부품 확률
    # --------------------------------------------------

    cv2.putText(

        frame,

        f"Target Confidence: "
        f"{average_confidence * 100:.1f}%",

        (30, 80),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.7,

        (0, 255, 0),

        2
    )


    # --------------------------------------------------
    # 다른 물건 확률
    # --------------------------------------------------

    cv2.putText(

        frame,

        f"Other: "
        f"{other_confidence * 100:.1f}%",

        (30, 115),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.7,

        (0, 0, 255),

        2
    )


    # --------------------------------------------------
    # 정답 횟수
    # --------------------------------------------------

    cv2.putText(

        frame,

        f"Correct: "
        f"{correct_count}/{required_count}",

        (30, 155),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.8,

        (255, 255, 255),

        2
    )


    # --------------------------------------------------
    # 안내 문구
    # --------------------------------------------------

    cv2.putText(

        frame,

        "Put object inside the box",

        (30, height - 30),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.7,

        (255, 255, 255),

        2
    )


    # ==================================================
    # 정답
    # ==================================================

    if correct_count >= required_count:

        completed_objects.append(
            target
        )


        print()

        print("🎉 정답!")

        print(
            f"찾은 물건: {target}"
        )

        print()


        # --------------------------------------------------
        # 현재 물건 제거
        # --------------------------------------------------

        remaining_objects.remove(
            target
        )


        # --------------------------------------------------
        # 모든 물건을 찾았는지 확인
        # --------------------------------------------------

        if len(remaining_objects) == 0:

            cv2.putText(

                frame,

                "GAME CLEAR!",

                (80, 220),

                cv2.FONT_HERSHEY_SIMPLEX,

                1.2,

                (0, 255, 0),

                3
            )


            cv2.imshow(
                "Object Game",
                frame
            )


            cv2.waitKey(2000)


            print("================================")

            print("       🎉 게임 클리어! 🎉")

            print("================================")

            print()

            print("찾은 물건:")

            print(
                completed_objects
            )


            break


        # ==================================================
        # 다음 게임
        # ==================================================

        target = random.choice(
            remaining_objects
        )


        # --------------------------------------------------
        # 다음 부품 모델 불러오기
        # --------------------------------------------------

        model = load_object_model(
            target
        )


        # --------------------------------------------------
        # 초기화
        # --------------------------------------------------

        correct_count = 0

        probability_history.clear()


        print("================================")

        print("🎯 다음 게임!")

        print(
            f"찾아야 하는 물건: {target}"
        )

        print("================================")


        continue


    # --------------------------------------------------
    # 웹캠 화면 출력
    # --------------------------------------------------

    cv2.imshow(
        "Object Game",
        frame
    )


    # --------------------------------------------------
    # Q로 종료
    # --------------------------------------------------

    if cv2.waitKey(1) & 0xFF == ord("q"):

        print()

        print(
            "게임을 종료했습니다."
        )

        break

QFontDatabase: Cannot find font directory /home/user14/Documents/GitHub/DeepLearningProject/.venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/user14/Documents/GitHub/DeepLearningProject/.venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/user14/Documents/GitHub/DeepLearningProject/.venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/user14/Documents/GitHub/DeepLearningProject/.venv/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://de


🎉 정답!
찾은 물건: nipper

🎯 다음 게임!
찾아야 하는 물건: pen

🎉 정답!
찾은 물건: pen

🎯 다음 게임!
찾아야 하는 물건: wire_stripper

게임을 종료했습니다.


## 셀 10 — 웹캠 종료

In [10]:
# ==================================================
# 10. 웹캠 종료
# ==================================================

camera.release()

cv2.destroyAllWindows()


print()

print("웹캠 종료")


웹캠 종료


In [11]:
import cv2

camera = cv2.VideoCapture(0)

if not camera.isOpened():
    print("❌ 카메라를 열 수 없습니다.")
else:
    print("✅ 카메라 시작")

while True:

    ret, frame = camera.read()

    if not ret:
        print("❌ 카메라 영상을 읽을 수 없습니다.")
        break

    cv2.imshow(
        "Camera Test",
        frame
    )

    key = cv2.waitKey(1) & 0xFF

    # Q를 누르면 종료
    if key == ord("q"):
        break


camera.release()

cv2.destroyAllWindows()

print("카메라 종료")

✅ 카메라 시작
카메라 종료
